In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

d:\Github-Learning\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

groq_model = ChatGroq(model="llama-3.3-70b-versatile") 
agent = create_agent(
    model=groq_model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c3537e02-c740-4ede-b4a9-37ac58d27205'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '161gtbb2f', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.054719212, 'completion_tokens_details': None, 'prompt_time': 0.037070356, 'prompt_tokens_details': None, 'queue_time': 0.049495424, 'total_time': 0.091789568}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eca44-bb7a-7ff2-b355-1177d30fc333-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'},

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='64b6921f-8135-42fe-a406-e52388795078'),
              AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 244, 'total_tokens': 269, 'completion_time': 0.071431487, 'completion_tokens_details': None, 'prompt_time': 0.014230556, 'prompt_tokens_details': None, 'queue_time': 0.160803204, 'total_time': 0.085662043}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eca44-d2ba-73f1-a4f1-c379d03c01f5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 244, 'output_tokens': 25, 'total_tokens': 269})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

from langchain_groq import ChatGroq

groq_model = ChatGroq(model="llama-3.3-70b-versatile") 
agent = create_agent(
    model=groq_model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='da02c4f0-ac60-4426-ad1b-1e97a5e4cbb4'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '80xb6wxtc', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 297, 'total_tokens': 316, 'completion_time': 0.047251137, 'completion_tokens_details': None, 'prompt_time': 0.030831494, 'prompt_tokens_details': None, 'queue_time': 0.049307756, 'total_time': 0.078082631}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eca45-07f3-7f23-8158-ce5580d7472e-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'},

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='da02c4f0-ac60-4426-ad1b-1e97a5e4cbb4'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '80xb6wxtc', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 297, 'total_tokens': 316, 'completion_time': 0.047251137, 'completion_tokens_details': None, 'prompt_time': 0.030831494, 'prompt_tokens_details': None, 'queue_time': 0.049307756, 'total_time': 0.078082631}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eca45-07f3-7f23-8158-ce5580d7472e-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'},